# LAB·J3 · Own the derivative

**Hardware:** any machine. ~2 h.

`jax.grad` differentiates the exact program you traced, not an approximation of it. Most of the time that is precisely what you want. This lab is about the two moments it is not enough on its own: when the automatic rule blows up numerically, and when you want a different trade between memory and compute than the one autodiff picked by default.

Predict before you run. Every reveal in this lab follows an empty prediction cell.


In [ ]:
import jax
import jax.numpy as jnp


## A gradient that should be easy

`log1pexp(x) = log(1 + exp(x))` is a softplus: smooth, monotonic, nothing exotic about it mathematically. Its derivative is the sigmoid function. Write it the obvious way and let `jax.grad` differentiate it automatically.

Predict what `jax.grad(log1pexp_naive)(100.0)` prints. The true derivative, sigmoid(100), is a number extremely close to 1.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

def log1pexp_naive(x):
    return jnp.log1p(jnp.exp(x))

print(jax.grad(log1pexp_naive)(100.0))


## Where the naive rule breaks

The naive version prints `nan`, not a number close to 1. `jnp.exp(100.0)` overflows float32: the largest float32 value is a little over $3.4 \times 10^{38}$, and $e^{100}$ is around $2.7 \times 10^{43}$, so the forward pass already carries an `inf` into `log1p`. The backward pass then needs $\frac{1}{1 + e^{100}} \times e^{100}$, and autodiff computes that literally: `1 / (1 + inf)` is `0`, and `0 * inf` is `nan`.

Nothing about this is a bug in JAX. The automatic rule differentiated the program exactly as written; the program itself loses precision before any gradient gets computed. Fixing this means supplying a gradient that never routes through the overflowing intermediate at all.


## Writing the gradient yourself

`jax.custom_vjp` lets you keep the forward computation exactly as written and replace only the backward rule with one you supply. For `log1pexp`, the stable backward rule is the sigmoid function itself: mathematically identical to what autodiff would compute if nothing overflowed, computed directly instead of derived through the overflowing path.

Predict what `jax.grad(log1pexp)(100.0)` prints once the custom rule is in place.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

@jax.custom_vjp
def log1pexp(x):
    return jnp.log1p(jnp.exp(x))

def fwd(x):
    return log1pexp(x), jax.nn.sigmoid(x)   # residual: the stable gradient

def bwd(sig, ct):
    return (ct * sig,)

log1pexp.defvjp(fwd, bwd)
print(jax.grad(log1pexp)(100.0))   # 1.0; the automatic rule returns nan here


## Why the fix works

`fwd` still computes `log1pexp` the same overflow-prone way, but it also saves a residual: `jax.nn.sigmoid(x)`, computed through its own numerically stable path rather than derived from the forward pass's intermediates. `bwd` never touches `exp` or `log1p` again; it just multiplies the incoming cotangent by that residual. The forward computation you traced and the gradient you get no longer share a numerical path, and that separation is exactly why the second one survives an input the first one cannot.

`jax.grad(log1pexp)(100.0)` prints `1.0`. That is not an approximation of the true derivative; sigmoid(100) really is 1.0 at float32 precision.


## Trading compute for memory

Reverse-mode autodiff pays for its cost model with memory: the backward pass needs every intermediate value the forward pass produced, held until the backward pass consumes it. `jax.checkpoint`, remat, offers a different trade: forget the intermediates from a block, and recompute that block's forward pass again during the backward pass instead of storing it.

The block below runs `tanh(x @ W)` eight times in a row. Predict whether `jax.grad` of the plain version and `jax.grad` of the checkpointed version agree, to floating-point tolerance.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

W = jnp.ones((256, 256)) * 0.01

def block(x):
    for _ in range(8):
        x = jnp.tanh(x @ W)
    return x.sum()

g_full = jax.grad(block)(jnp.ones(256))
g_remat = jax.grad(jax.checkpoint(block))(jnp.ones(256))
print(jnp.allclose(g_full, g_remat))   # True; the remat pass stored no intermediates


## Same gradient, different memory bill

`jnp.allclose(g_full, g_remat)` prints `True`. Both calls compute the same mathematical gradient; they differ only in what the backward pass does to get there. `g_full` carries eight layers of stored activations forward into the backward pass. `g_remat` drops each layer's activations as soon as the forward pass moves past them, then recomputes that layer's forward pass again when the backward pass needs it. The FLOPs go up by roughly one extra forward pass; the peak memory goes down by the cost of the residuals no longer stored. Flash attention makes exactly this trade, at a far larger scale than an eight-layer block.


## One gradient per example

`vmap` and `grad` compose: `vmap(grad(loss), in_axes=(None, 0, 0))` differentiates `loss` with respect to its first argument, once per example along the batched axes of the other two, and returns the whole batch of gradients as one array. Nothing here is a loop in Python; it is one program, batched.

Predict the shape of `per_sample` below, given `w` has shape `(3,)` and `xs`/`ys` each batch 32 examples.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

def loss(w, x, y):
    return (x @ w - y) ** 2

w = jnp.ones(3)
xs = jnp.ones((32, 3))
ys = jnp.zeros(32)
per_sample = jax.vmap(jax.grad(loss), in_axes=(None, 0, 0))(w, xs, ys)
print(per_sample.shape)   # (32, 3): one gradient per example


## What the shape tells you

`per_sample.shape` is `(32, 3)`: one three-element gradient per example, at the cost of one batched matmul, not thirty-two separate calls to `grad`. This is the shape per-example gradient clipping needs. Differential privacy and some robust-training methods clip each example's gradient to a maximum norm before averaging, precisely so one outlier example cannot dominate the batch's update; without `vmap`, getting one gradient per example would mean giving up the batched computation entirely.


## Exercise

Write `clipped_mean_grad(w, xs, ys, max_norm)`: compute `per_sample` as above, clip each row to `max_norm` by its L2 norm (scale a row down only if its norm exceeds `max_norm`), then return the mean over examples. Ten lines or fewer. Verify it against the unclipped mean when every gradient's norm already sits under `max_norm`; the two should match.


## Mark it run

You have watched an automatic gradient fail on a real numerical edge case, replaced it with one you wrote yourself, proven that trading compute for memory does not change the answer, and turned per-sample gradients into a single batched program. Go back to chapters 4 and 5 on the chapter page and tick LAB·J3 as run.
